In [5]:
import pandas as pd
import torch

In [6]:
df=pd.read_csv("/content/Dataset for rnn.csv")
df.head(10)
df.dtype()

FileNotFoundError: [Errno 2] No such file or directory: '/content/Dataset for rnn.csv'

In [ ]:
# spliting data and removing unwanted things (tokenize)
def  tokenize(text):
  text=text.lower()
  text=text.replace("?","")
  text=text.replace("'","")
  return text.split()

tokenize("What is the square root of 64?	")


In [ ]:
# finding uniqe text and give them a token

vocab={"<UNK>":0}

def build_vocab(row):
  # print(tokenize(row["question"]),tokenize(row["answer"]))
  tokenized_quetion=tokenize(row["question"])
  tokenized_answer=tokenize(row["answer"])
  merzed_token=tokenized_quetion+tokenized_answer
  print(merzed_token)

  for token in merzed_token:
    if token not in vocab:
      vocab[token]=len(vocab)


In [ ]:
df.apply(build_vocab,axis=1)

In [ ]:
vocab

In [ ]:
# len(vocab)
# tokenize("hello my is dipesh ?")


In [ ]:
# now converting a text into numbers

def text_to_indecies(text,vocab):
  indecies=[]

  for token in tokenize(text):

    if token in vocab:
      indecies.append(vocab[token])
    else:
      indecies.append(vocab['<UNK>'])
  return indecies

In [ ]:
text_to_indecies("What is the boiling point of water in Celsius?",vocab)

In [ ]:
from torch.utils.data import Dataset,DataLoader



In [ ]:
class QADataset(Dataset):

  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numeric_question=text_to_indecies(self.df.iloc[index]['question'],vocab)
    numeric_answer=text_to_indecies(self.df.iloc[index]["answer"],vocab)

    return torch.tensor(numeric_question),torch.tensor(numeric_answer)

In [ ]:
data=QADataset(df,vocab)
data.__getitem__(10)

In [ ]:
dataloader=DataLoader(data,batch_size=1,shuffle=True)

In [ ]:
for quetion , answer in dataloader:
  print(quetion)
  print(answer)

In [ ]:
import torch.nn as nn
# model define
class MyRnn(nn.Module):

  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64)
    self.fc=nn.Linear(64,vocab_size)

  def forward(self,quetion):
    embedded_quetoin=self.embedding(quetion)
    hidden,final=self.rnn(embedded_quetoin)
    output=self.fc(final)
    return output

In [ ]:
epoches=100
lr=0.001
criteria=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=lr)


In [ ]:
model=MyRnn(len(vocab))

In [ ]:
# training loop
i=0
for epoch in range(epoches):
  total_loss=0
  for quetion,answer in dataloader:
    optimizer.zero_grad()

    y_pred=model(quetion)

    loss=criteria(y_pred,output)

    loss.backward()

    optimizer.step()

    total_loss += total_loss
    i  +=1
  print("loss in {ith }")